# 04 — Train frontal-eddy U-Net (Colab GPU)

Segmentation: (SSHA, ADT, |∇ADT|) 128×128 tiles → frontal-eddy mask. Targets = PET
speed contours of hand-confirmed frontal eddies (`scripts/build_masks.py`).

Temporal split: **train 1–30** (fit) / **val 31–35** (model selection) / **cal 36–42**
(conformal calibration, untouched by any modeling choice) / **test 43–50** (final eval).

**Colab setup:** zip `out/aoi5/training/` locally → upload to Drive. Repo code comes
from GitHub clone. Known v1 caveats: PET-missed frontals are background in targets;
per-tile inpaint seams in inputs.

In [ ]:
import sys, os
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    !pip -q install torch --index-url https://download.pytorch.org/whl/cu121 2>/dev/null || pip -q install torch
    # repo code: clone once (private repo -> use a token, or upload src/ to Drive)
    if not Path("swot-eddy-ml").exists():
        !git clone https://github.com/Originaljsx/swot-eddy-ml.git
    sys.path.insert(0, "swot-eddy-ml/src")
    TRAIN_DIR = "/content/drive/MyDrive/swot-eddy-ml/training"   # unzip target
    OUT_DIR = "/content/drive/MyDrive/swot-eddy-ml/model"
else:
    sys.path.insert(0, str(Path.cwd().parent / "src"))
    TRAIN_DIR = str(Path.cwd().parent / "out/aoi5/training")
    OUT_DIR = str(Path.cwd().parent / "out/aoi5/model")

import torch
import numpy as np
from swot_eddy.unet import build_unet, TileDataset, dice_bce_loss

DEV = "cuda" if torch.cuda.is_available() else "cpu"
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
print("device:", DEV, "| training data:", TRAIN_DIR)

In [ ]:
from torch.utils.data import DataLoader

train_ds = TileDataset(TRAIN_DIR, "train", pos_frac=0.5, augment=True)
val_ds = TileDataset(TRAIN_DIR, "val", pos_frac=None)
print(f"train: {len(train_ds)} tiles ({len(train_ds.pos)} positive)")
print(f"val:   {len(val_ds)} tiles ({len(val_ds.pos)} positive)")
train_dl = DataLoader(train_ds, batch_size=32, num_workers=2, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=32, num_workers=2)

In [ ]:
def val_metrics(model):
    model.eval()
    inter = union = tp = fp = fn = 0
    with torch.no_grad():
        for x, y in val_dl:
            p = (torch.sigmoid(model(x.to(DEV))) > 0.5).float().cpu()
            inter += (p * y).sum().item(); union += ((p + y) > 0).sum().item()
            tp += ((p == 1) & (y == 1)).sum().item()
            fp += ((p == 1) & (y == 0)).sum().item()
            fn += ((p == 0) & (y == 1)).sum().item()
    iou = inter / max(union, 1)
    return iou, tp / max(tp + fp, 1), tp / max(tp + fn, 1)

model = build_unet().to(DEV)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=30)
best_iou, EPOCHS = 0.0, 30
for ep in range(EPOCHS):
    model.train()
    tot = 0.0
    for x, y in train_dl:
        opt.zero_grad()
        loss = dice_bce_loss(model(x.to(DEV)), y.to(DEV))
        loss.backward(); opt.step()
        tot += loss.item()
    sched.step()
    iou, prec, rec = val_metrics(model)
    star = ""
    if iou > best_iou:
        best_iou = iou
        torch.save(model.state_dict(), f"{OUT_DIR}/unet_frontal.pt")
        star = " *saved"
    print(f"ep {ep+1:02d}  loss {tot/len(train_dl):.4f}  "
          f"val IoU {iou:.3f} P {prec:.3f} R {rec:.3f}{star}")

In [ ]:
# eddy-LEVEL evaluation on the held-out TEST split (cycles 43-50): a predicted blob
# counts as a hit if it overlaps a true frontal mask (IoU-free, detection-style).
from scipy import ndimage
import pandas as pd
from swot_eddy.unet import normalize, pass_tiles, TILE

model.load_state_dict(torch.load(f"{OUT_DIR}/unet_frontal.pt", map_location=DEV))
model.eval()
man = pd.read_csv(f"{TRAIN_DIR}/training_manifest.csv", dtype={"cycle": str, "pass": str})
test = man[(man.split == "test") & (man.status == "ok")]
TP = FP = FN = 0
for c, p in zip(test.cycle, test["pass"]):
    with np.load(f"{TRAIN_DIR}/{c}_{p}.npz") as d:
        x, y = d["x"].astype(np.float32), d["y"]
    L = x.shape[1]
    prob = np.zeros((L, 128), np.float32); hits = np.zeros((L, 128), np.float32)
    with torch.no_grad():
        for s, xt in pass_tiles(x):
            pr = torch.sigmoid(model(torch.from_numpy(normalize(xt))[None].to(DEV)))[0, 0].cpu().numpy()
            e = min(s + TILE, L)
            prob[s:e] += pr[: e - s]; hits[s:e] += 1
    pred = (prob / np.maximum(hits, 1)) > 0.5
    pl, pn = ndimage.label(pred)
    tl, tn = ndimage.label(y)
    hit_true = set(np.unique(tl[pred])) - {0}
    hit_pred = set(np.unique(pl[y > 0])) - {0}
    TP += len(hit_true); FN += tn - len(hit_true); FP += pn - len(hit_pred)
print(f"TEST eddy-level: precision {TP/max(TP+FP,1):.2f}  recall {TP/max(TP+FN,1):.2f}  "
      f"(TP {TP} FP {FP} FN {FN})")
print("NOTE: recall is vs PET-derived truth — PET-missed frontals aren't counted.")

In [ ]:
# CONFORMAL calibration (cal split, cycles 36-42): pick detection threshold lam_hat with
# a distribution-free guarantee that a new frontal eddy is missed with prob <= ALPHA.
# Score per true eddy = max predicted probability inside its mask. lam_hat = the
# floor(ALPHA*(n+1)-1)/n lower empirical quantile of the scores (split-conformal bound).
# Caveat (report honestly): guarantee assumes eddy exchangeability; eddies from the same
# cycle are weakly dependent, so coverage is approximate at the cycle level.
ALPHA = 0.10

def eddy_scores(split):
    scores = []
    sub = man[(man.split == split) & (man.status == "ok") & (man.n_frontal > 0)]
    for c, p in zip(sub.cycle, sub["pass"]):
        with np.load(f"{TRAIN_DIR}/{c}_{p}.npz") as d:
            x, y = d["x"].astype(np.float32), d["y"]
        L = x.shape[1]
        prob = np.zeros((L, 128), np.float32); hits = np.zeros((L, 128), np.float32)
        with torch.no_grad():
            for s, xt in pass_tiles(x):
                pr = torch.sigmoid(model(torch.from_numpy(normalize(xt))[None].to(DEV)))[0, 0].cpu().numpy()
                e = min(s + TILE, L)
                prob[s:e] += pr[: e - s]; hits[s:e] += 1
        prob /= np.maximum(hits, 1)
        tl, tn = ndimage.label(y)
        for k in range(1, tn + 1):
            scores.append(float(prob[tl == k].max()))
    return np.array(scores)

cal_scores = eddy_scores("cal")
n = len(cal_scores)
k = int(np.floor(ALPHA * (n + 1) - 1))
lam_hat = float(np.sort(cal_scores)[max(k, 0)])
print(f"cal eddies: {n} | lam_hat = {lam_hat:.3f} (target miss rate <= {ALPHA:.0%})")

# empirical check on TEST: fraction of true frontal eddies with max-prob >= lam_hat
test_scores = eddy_scores("test")
print(f"test coverage: {np.mean(test_scores >= lam_hat):.1%} of {len(test_scores)} "
      f"frontal eddies detected at lam_hat (guarantee: >= {1-ALPHA:.0%})")
np.save(f"{OUT_DIR}/lam_hat.npy", np.array([lam_hat, ALPHA]))